<a href="https://colab.research.google.com/github/kantathamangjam-lgtm/GenAI/blob/main/StudyBuddy_multiAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-genai pypdf

from google import genai
from google.genai import types
import os
from google.colab import userdata

# Retrieve API key from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
#MODEL_ID = "gemini-3.6-flash"
MODEL_ID = "gemini-3.8-flash" # for faster execution

In [2]:
# Option A: Upload local PDF to Colab session
from google.colab import files
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

# Option B: Use a file path or URI
# pdf_file = client.files.upload(file=pdf_filename)

Saving sample.pdf to sample (1).pdf


In [3]:
def reader_agent(pdf_path, model_id=MODEL_ID):
    """Agent 1: Reads the PDF and extracts structured raw insights."""
    uploaded_pdf = client.files.upload(file=pdf_path)

    prompt = """
    You are a meticulous Document Reader Agent.
    Analyze this PDF thoroughly. Extract the core thesis, main arguments,
    key quantitative data/tables, and section-by-section breakdown.
    """

    response = client.models.generate_content(
        model=model_id,
        contents=[uploaded_pdf, prompt]
    )
    return response.text

def synthesizer_agent(raw_extraction, model_id=MODEL_ID):
    """Agent 2: Refines and synthesizes the raw notes into an executive summary."""
    prompt = f"""
    You are an Executive Editor Agent. Review the following raw document extraction
    provided by the Reader Agent and synthesize it into:
    1. A 3-sentence high-level overview.
    2. Key bulleted takeaways (grouped by theme).
    3. Critical conclusions or implications.

    Raw Notes:
    {raw_extraction}
    """

    response = client.models.generate_content(
        model=model_id,
        contents=prompt
    )
    return response.text

# Execute the multi-agent workflow
print("--- [Agent 1: Reading & Extracting] ---")
raw_notes = reader_agent(pdf_filename)

print("\n--- [Agent 2: Synthesizing Summary] ---")
final_summary = synthesizer_agent(raw_notes)
print(final_summary)


--- [Agent 1: Reading & Extracting] ---

--- [Agent 2: Synthesizing Summary] ---


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [4]:
#Quizz
def generate_quiz(pdf_text: str) -> str:
  """Sends text to Gemini to generate a 5-question multiple choice quiz."""
  prompt = (
      "Based on the following text from a PDF, generate a 5-question"
      " multiple-choice quiz. Provide 4 options (A, B, C, D) for each question"
      " and include an answer key at the very end.\n\nText:\n"
      f"{pdf_text[:10000]}"  # Truncate to fit context if very large
  )

  response = client.models.generate_content(
      model="gemini-3.6-flash",
      contents=prompt,
  )
  return response.text

print("Generating quiz with Gemini...")
quiz = generate_quiz(raw_notes)

print("\n--- Generated Quiz ---\n")
print(quiz)


Generating quiz with Gemini...

--- Generated Quiz ---

Here is a 5-question multiple-choice quiz based on the provided document:

### **Solar System Quiz**

**1. How long does it take for the solar system to complete one orbit around the center of the Milky Way galaxy?**  
A) 225 thousand years  
B) 225 million years  
C) 1 billion years  
D) 2.5 billion years  

**2. How are the eight official planets distributed across the three planetary classifications?**  
A) 4 Terrestrial, 2 Gas Giants, 2 Ice Giants  
B) 3 Terrestrial, 3 Gas Giants, 2 Ice Giants  
C) 4 Terrestrial, 3 Gas Giants, 1 Ice Giant  
D) 2 Terrestrial, 4 Gas Giants, 2 Ice Giants  

**3. Which of the following icy fluids characterize the mantles surrounding the rocky cores of Uranus and Neptune?**  
A) Nitrogen, oxygen, and carbon dioxide  
B) Hydrogen, helium, and iron oxide  
C) Water, methane, and ammonia  
D) Argon, carbon monoxide, and methane  

**4. Which planet is described as having an extreme axial tilt that cau